# Kaggle fine-tune: Solidity hunter (QLoRA on `splits/train_source.jsonl`)

One-click training notebook for the [smartcontractshacking](https://github.com/lipon101/smartcontractshacking) corpus.
It consumes **`splits/train_source.jsonl` directly** — including the GitHub 100 MB byte-split `.partNN` layout —
and formats every record into the **strict-JSON answer format** already defined by the repo's
[`scripts/eval_harness.py`](https://github.com/lipon101/smartcontractshacking/blob/main/scripts/eval_harness.py)
(SYSTEM_PROMPT + `extract_json`), so the fine-tuned model speaks exactly the dialect the Echidna gate expects.
Matched items from the [Findings Checklist](https://github.com/lipon101/smartcontractshacking/blob/main/docs/findings-checklist.md)
are injected into the user prompt as reference context (`CHECKLIST_CONTEXT`).

**Answer contract (verbatim from `eval_harness.py`):**

```json
{"vulnerable": true/false, "vuln_type": "...", "severity": "critical|high|medium|low|gas",
 "poc": "<step-by-step exploit sequence>", "fix": "<what the fix is and why>",
 "patched_function": "<the FULL replacement function including its signature, or null if not vulnerable>"}
```

**How to run on Kaggle (one click):**
1. Create a Kaggle Dataset from the repo's `splits/` folder (upload the `train_source.jsonl.part00..03` + `eval_source.jsonl` files as-is; the loader reassembles them). Name it anything — the notebook auto-discovers `/kaggle/input/*/splits`.
2. New Notebook → **Add Input** → your dataset → Runtime: **GPU T4 x2** (16 GB).
3. **Run All.** Training (~1–2 h for Qwen3.5-9B QLoRA, 2 epochs) then an eval cell reports strict-JSON accuracy on the held-out `eval_source.jsonl` split.

**Model** (verified on HF, 2026-08-03; see `docs/qwen36-exact-links-and-tools.md`):
- Default `Qwen/Qwen3.5-9B` — fits the Kaggle T4 16 GB (docs: 16 GB fallback).
- `Qwen/Qwen3.6-27B` (Apache-2.0, 24 GB min) or the repo's stated target `samscrack/Qwen3.6-Solidity-27B` / `crichalchemist/Qwen3.6-Solidity-27B` — set `MODEL_ID` below and use a ≥24 GB GPU.

**After training:** `vps_setup.sh` serves the GGUF via Ollama; `eval_harness.py --model hunter --contract x.sol --function f` runs the differential-fuzz gate.


In [ ]:
# 1) Environment — Kaggle images already ship torch/transformers; install the rest.
#    NOTE: never install CUDA wheels on Kaggle; the prebuilt image has them.
import subprocess, sys

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

_pip("transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets")

import torch, transformers, accelerate
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| accelerate", accelerate.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")


In [ ]:
# 2) Configuration — model, paths, data policy, training hyperparameters.

import os, glob, json, hashlib, random
from pathlib import Path
from collections import Counter

SEED = 42
random.seed(SEED)

# ---------------- corpus paths ----------------
# Auto-discovery order: $CORPUS_DIR -> /kaggle/input/<any-dataset>/splits
# -> /kaggle/input/<any-dataset>/ (files loose in dataset root) -> local clone.
DATA_DIR = os.environ.get("CORPUS_DIR", "")
if not DATA_DIR:
    candidates = sorted(glob.glob("/kaggle/input/*/splits")) + sorted(glob.glob("/kaggle/input/*/"))
    candidates += [str(Path.cwd() / "splits"), str(Path.cwd().parent / "splits")]
    for c in candidates:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "train_source.jsonl*")):
            DATA_DIR = c
            break
assert DATA_DIR, "Could not find the corpus. Set CORPUS_DIR=<path> or mount the Kaggle dataset."
print("DATA_DIR =", DATA_DIR)

TRAIN_GLOB = "train_source.jsonl*"   # 4 byte-split parts OR one reassembled file — both work
EVAL_FILE  = "eval_source.jsonl"

# SHA256 of the reassembled originals (README.md) — the loader verifies byte-exact integrity.
EXPECTED_SHA = {
    "train_source.jsonl": "a8fea891370879fa143ba15e4a12ff586fa51069318dd4d9ad31ad03e999012e",
    "eval_source.jsonl":  "312c14932754a9e56c0755d9196b1b4289fd8ca740eb147c18bad19deb8e6b40",
}
VERIFY_SHA = True
MAX_ROWS   = None          # optional row cap for local testing only

# ---------------- model / fine-tune ----------------
# T4 (16 GB) one-click default: Qwen3.5-9B. 27B variants need >=24 GB VRAM:
#   "Qwen/Qwen3.6-27B"                      base model (Apache-2.0, the docs pick)
#   "samscrack/Qwen3.6-Solidity-27B"        repo's stated Kaggle target (kaggle_ready_report.json)
#   "crichalchemist/Qwen3.6-Solidity-27B"   leaderboard-topping Solidity checkpoint
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3.5-9B")

CHECKLIST_CONTEXT = True   # inject matched Findings Checklist items into the user prompt
                           # (items + patterns are embedded in this notebook; see docs/findings_checklist.json)

MAX_SEQ_LEN       = 4096   # context cap; source/audit text is truncated to fit
MAX_CONTEXT_CHARS = 6000   # chars of contract source / audit narrative shown to the model
MAX_NEW_TOKENS    = 512    # answer length cap during eval generation

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

EPOCHS           = 2
LR               = 2e-4
PER_DEVICE_BATCH = 1
GRAD_ACCUM       = 16      # effective batch = 1 * 16 = 16
WARMUP_RATIO     = 0.03
EVAL_MAX_ROWS    = 400     # rows of eval_source used for validation metrics
EVAL_N_ROWS      = 50      # rows used for the post-training strict-JSON eval
OUTPUT_DIR       = "/kaggle/working/hunter-qlora"

print("MODEL_ID =", MODEL_ID, "| MAX_SEQ_LEN =", MAX_SEQ_LEN)


In [ ]:
# 3) Loader — consume splits/train_source.jsonl DIRECTLY (byte-split parts included).
#    The .partNN files are byte-splits, NOT line-splits: 3 records straddle part
#    boundaries. We therefore stream the concatenated byte stream (cat part* in
#    sorted order), which is the only correct way to parse them, and verify the
#    published SHA256.

def cat_lines(files):
    '''Yield decoded lines of the byte-concatenated stream (true `cat part*` semantics:
    a record straddling a part boundary is reassembled before line-splitting).'''
    buf = b""
    for f in files:
        with open(f, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                buf += chunk
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    yield line.decode("utf-8")
    if buf:
        yield buf.decode("utf-8")

def load_jsonl_parts(glob_pattern, verify_sha=None):
    '''Reassemble byte-split parts (cat part* in sorted order) and parse one JSON per line.'''
    files = sorted(glob.glob(os.path.join(DATA_DIR, glob_pattern)))
    assert files, f"no files match {DATA_DIR}/{glob_pattern}"
    print(f"reading {len(files)} file(s): {[os.path.basename(f) for f in files]}")

    if verify_sha:
        h = hashlib.sha256()
        for f in files:
            with open(f, "rb") as fh:
                for chunk in iter(lambda: fh.read(1 << 20), b""):
                    h.update(chunk)
        got = h.hexdigest()
        print(f"sha256(concat) = {got}")
        assert got == verify_sha, f"SHA256 MISMATCH — expected {verify_sha}, got {got}"

    rows = []
    for line in cat_lines(files):
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
        if MAX_ROWS and len(rows) >= MAX_ROWS:
            return rows
    return rows

train_rows = load_jsonl_parts(TRAIN_GLOB, EXPECTED_SHA["train_source.jsonl"] if VERIFY_SHA else None)
eval_rows  = load_jsonl_parts(EVAL_FILE,  EXPECTED_SHA["eval_source.jsonl"]  if VERIFY_SHA else None)

print(f"train = {len(train_rows)} records | eval = {len(eval_rows)} records")
print("train keys:", sorted(train_rows[0].keys()))


In [ ]:
# 4) Leakage & stats — the repo ships a protocol-exclusive split; never re-split randomly.
#    (Re-splitting train_source.jsonl at random would leak protocols into validation.)

train_ids, eval_ids   = {r["id"] for r in train_rows}, {r["id"] for r in eval_rows}
train_prot, eval_prot = {r["protocol"] for r in train_rows}, {r["protocol"] for r in eval_rows}
assert train_ids.isdisjoint(eval_ids),   "FAIL: id overlap between train and eval!"
assert train_prot.isdisjoint(eval_prot), "FAIL: protocol overlap between train and eval!"
print(f"id overlap       = {len(train_ids & eval_ids)}")
print(f"protocol overlap = {len(train_prot & eval_prot)}  (train {len(train_prot)} / eval {len(eval_prot)})")

print("train severity:", dict(Counter(r["severity"] for r in train_rows)))
print("eval  severity:", dict(Counter(r["severity"] for r in eval_rows)))
print(f"train with source   : {sum(1 for r in train_rows if r['source'])}/{len(train_rows)}")
print(f"train with function : {sum(1 for r in train_rows if r['function'])}/{len(train_rows)}")
print(f"train empty vuln_type: {sum(1 for r in train_rows if not r['vuln_type'].strip())}")


In [ ]:
# 5) Format — every record becomes a chat triple ending in the STRICT-JSON answer.
#    SYSTEM_PROMPT mirrors scripts/eval_harness.py verbatim (the repo's single
#    source of truth for the answer contract). Severity is normalized from the
#    corpus enum (LOW/MEDIUM/HIGH/GAS) to the harness enum (low/medium/high/gas).
#    Answers are serialized with json.dumps(ensure_ascii=False) — never hand-built.
#    If CHECKLIST_CONTEXT, matched Findings Checklist items are appended to the
#    user prompt as reference context (embedded from docs/findings_checklist.json
#    + scripts/checklist_map.py at notebook build time).

# ---- embedded checklist data (self-contained for Kaggle) ----
import re as _re
CHECKLIST_ITEMS = json.loads(r'''{"Denial-Of-Service (DoS) Attack": [{"title": "Is the withdrawal pattern followed to prevent denial of service?", "description": "To prevent denial of service attacks during withdrawals, it's critical to follow the withdrawal pattern best practices - pull based approach.", "remediation": "Implement withdrawal pattern best practices to ensure that contract behavior remains predictable and robust against denial of service attacks."}, {"title": "Is there a minimum transaction amount enforced?", "description": "Enforcing a minimum transaction amount can prevent attackers from clogging the network with zero amount or dust transactions.", "remediation": "Disallow transactions below a certain threshold to maintain efficiency and prevent denial of service through dust spamming."}, {"title": "How does the protocol handle tokens with blacklisting functionality?", "description": "Tokens with blacklisting capabilities, such as USDC, can pose unique risks and challenges to protocols.", "remediation": "Account for the possibility of blacklisting within token protocols to ensure continued functionality even if certain addresses are blacklisted."}, {"title": "Can forcing the protocol to process a queue lead to DOS?", "description": "Forcing protocols to process queues, like a queue of dust withdrawals, can be exploited to cause a denial of service.", "remediation": "Design queue processing in a manner that is resilient to spam and cannot be exploited to cause denial of service."}, {"title": "What happens with low decimal tokens that might cause DOS?", "description": "Tokens with low decimals can present issues where the transaction process fails due to rounding to zero amounts.", "remediation": "Implement logic to handle low decimal tokens in a way that prevents the transaction process from breaking due to insufficient token amounts."}, {"title": "Does the protocol handle external contract interactions safely?", "description": "Protocols must handle interactions with external contracts in a way that does not compromise their functionality if external dependencies fail.", "remediation": "Ensure robust handling of external contract interactions to maintain protocol integrity regardless of external contract performance."}], "Donation Attack": [{"title": "Does the protocol rely on `balance` or `balanceOf` instead of internal accounting?", "description": "Attackers can manipulate the accounting by donating tokens.", "remediation": "Implement internal accounting instead of relying on `balanceOf` natively."}], "Front-running Attack": [{"title": "Are \"get-or-create\" patterns protected against front-running attacks?", "description": "Functions combining resource creation and interaction (like getOrCreateAndUse) are vulnerable to front-running attacks where attackers can create the resource with different parameters before the victim, potentially manipulating prices or conditions.", "remediation": "Separate creation and interaction into distinct transactions or implement robust protections (parameter validation, relative references instead of absolute values) to ensure safe operation regardless of creation timing."}, {"title": "Are two-transaction actions designed to be safe from frontrunning?", "description": "Actions that require two separate transactions may be at risk of frontrunning, where an attacker can intervene between the two calls.", "remediation": "Ensure critical actions that are split across multiple transactions cannot be interfered with by attackers. This can involve checks or locks between the transactions."}, {"title": "Can users maliciously cause others' transactions to revert by preempting with dust?", "description": "Attackers may cause legitimate transactions to fail by front-running with transactions of negligible amounts.", "remediation": "Implement checks to prevent transactions with non-material amounts from affecting the contract's state or execution flow."}, {"title": "Is the protocol using a properly user-bound commit-reveal scheme?", "description": "Sensitive on-chain actions can be exposed in the mempool, enabling frontrunning and information exploitation. Effective commit-reveal schemes must bind commitments to specific users and transactions.", "remediation": "Implement a two-phase process where users first commit a hash containing their address and all transaction parameters, then reveal actual actions after the commitment phase ends, preventing frontrunning and information leakage."}], "Griefing Attack": [{"title": "Is there an external function that relies on states that can be changed by others?", "description": "Malicious actors can prevent regular user transactions by making a slight change to the on-chain states.", "remediation": "Ensure normal user actions especially important actions like withdrawal and repayment are not disturbed by other actors."}, {"title": "Can the contract operations be manipulated with precise gas limit specifications?", "description": "Attackers can supply carefully calculated gas amounts to force specific execution paths in the contract, manipulating its behavior in unexpected ways.", "remediation": "Implement explicit gas checks before critical operations."}], "Miner Attack": [{"title": "Is block.timestamp used for time-sensitive operations?", "description": "Miners can manipulate block.timestamp by several seconds, potentially affecting time-dependent contract logic.", "remediation": "Use block.number instead of timestamps for critical timing operations or ensure manipulation tolerance is acceptable."}, {"title": "Is the contract using block properties like timestamp or difficulty for randomness generation?", "description": "Block properties (timestamp, difficulty) and other predictable values should not be used for randomness as they can be influenced or predicted by miners.", "remediation": "Use a secure randomness source like Chainlink VRF, commit-reveal schemes, or a provably fair randomization mechanism instead."}, {"title": "Is contract logic sensitive to transaction ordering?", "description": "Miners control transaction ordering and can exploit this for front-running, back-running, or sandwich attacks.", "remediation": "Implement protection by allowing users to specify acceptable results that revert transactions when breached."}], "Price Manipulation Attack": [{"title": "Is the price calculated by the ratio of token balances?", "description": "Price can be manipulated via flash loans or donations if it is derived from the ratio of token balances.", "remediation": "Use the Chainlink oracles for the asset prices."}, {"title": "Is the price calculated from DEX liquidity pool spot prices?", "description": "Spot price readings derived directly from DEX liquidity pools are vulnerable to manipulation through flash loans that can temporarily drain the pools.", "remediation": "Use TWAP (time-weighted average price) with appropriate time windows based on asset volatility and liquidity, or use reliable oracle solutions."}], "Reentrancy Attack": [{"title": "Is there any state change after interaction with an external contract?", "description": "Untrusted external contract calls could callback leading to unexpected results such as multiple withdrawals or out-of-order events.", "remediation": "Use check-effects-interactions pattern or reentrancy guards."}, {"title": "Is there a view function that can return a stale value during interactions?", "description": "Read-only reentrancy occurs when a view function, called during a reentrant execution, returns inaccurate data because the contract's state is temporarily inconsistent due to an ongoing external call, potentially misleading dependent protocols.", "remediation": "Apply the Check-Effects-Interactions pattern to prevent inconsistent state, and ensure the reentrancy guard state is not ENTERED for critical view functions to prevent returning stale data."}], "Replay Attack": [{"title": "Are there protections against replay attacks for failed transactions?", "description": "Failed transactions can be susceptible to replay attacks if not properly protected.", "remediation": "Implement nonce-based or other mechanisms to ensure that each transaction can only be executed once, preventing replay attacks, even if the transaction initially failed."}, {"title": "Is there protection against replaying signatures on different chains?", "description": "Signatures that are valid on one blockchain may be replayed on another, leading to potential security breaches.", "remediation": "Use chain-specific parameters, such as `block.chainid`, or domain separators as defined in EIP-712 to ensure signatures are only valid on the intended chain."}], "Rug Pull": [{"title": "Can the admin of the protocol pull assets from the protocol?", "description": "Some protocols grant an admin with a privilege of pulling assets directly from the protocol. In general, if there is an actor that can affect the user funds directly it must be reported.", "remediation": "Allow access to only the relevant parts of protocol funds, e.g. by tracking fees internally. Forcing a timelock on the admin actions can be another mitigation."}], "Sandwich Attack": [{"title": "Does the protocol have an explicit slippage protection on user interactions?", "description": "An attacker can monitor the mempool and puts two transactions before and after the user's transaction. For example, when an attacker spots a large trade, executes their own trade first to manipulate the price, and then profits by closing their position after the user's trade is executed.", "remediation": "Allow users to specify the minimum output amount and revert the transaction if it is not satisfied."}], "Sybil Attack": [{"title": "Is there a mechanism depending on the number of users?", "description": "It is very easy to trigger actions using a lot of alternative addresses on blockchain. Any quorum mechanism or utilization based rewarding system can be vulnerable to sybil attacks.", "remediation": "Do not rely on the number of users in quorum design."}]}''')
CHECKLIST_PATTERNS = json.loads(r'''{"Denial-Of-Service (DoS) Attack": ["\\bdos\\b", "denial of service", "denial-of-service", "withdraw(al)? pattern", "pull[- ]?based", "\\bdust", "queue", "low decimal", "blacklist", "grief", "out of gas", "block gas", "unbounded loop", "gas limit"], "Donation Attack": ["(?s)donat\\\\w*.{0,30}(attack|inflate|manipulat|tokens?|funds?|balance|pool|price|account)", "balanceof", "internal accounting"], "Front-running Attack": ["front[- ]?run", "mempool", "commit[- ]?reveal", "get[- ]?or[- ]?create", "preempt", "two[- ]transaction"], "Griefing Attack": ["grief", "gas limit", "force (a )?(revert|failure)", "prevent (the )?(withdrawal|repayment)"], "Miner Attack": ["block\\.timestamp", "\\btimestamp", "block\\.number", "randomness", "\\bminer", "transaction order", "tx order", "predictable"], "Price Manipulation Attack": ["\\bprice", "\\boracle", "\\btwap", "flash[- ]?loan", "spot price", "liquidity pool", "ratio of token balances", "manipulat"], "Reentrancy Attack": ["reentran", "callback", "check[- ]?effects[- ]?interactions", "state change after interaction", "read[- ]?only reentran"], "Replay Attack": ["replay", "\\bnonce", "chainid", "chain id", "eip-712", "signature"], "Rug Pull": ["rug", "pull (all |the )?(assets|funds|tokens)", "admin.*(pull|withdraw|steal)", "owner.*(pull|withdraw|steal)", "pull assets"], "Sandwich Attack": ["sandwich", "slippage", "min(imum)? (output|amount)", "minout", "min out"], "Sybil Attack": ["sybil", "quorum", "number of users", "unique users"]}''')
_CHECKLIST_RE = {cat: [_re.compile(p, _re.IGNORECASE) for p in pats]
                 for cat, pats in CHECKLIST_PATTERNS.items()}

def match_checklist_categories(*texts):
    '''Checklist categories whose patterns hit any of the given texts.'''
    hits = []
    for cat, res in _CHECKLIST_RE.items():
        for rx in res:
            if any(rx.search(t) for t in texts if t):
                hits.append(cat)
                break
    return hits

def checklist_context(vuln_type, audit_text="", max_cats=2, max_items=2):
    '''Compact checklist reference block for the matched categories ("" if none).'''
    cats = match_checklist_categories(vuln_type, (audit_text or "")[:2000])
    if not cats:
        return ""
    lines = []
    for cat in cats[:max_cats]:
        for it in CHECKLIST_ITEMS.get(cat, [])[:max_items]:
            lines.append(f"- [{cat}] {it['title']} — {it['remediation']}")
    return "Reference checklist:\n" + "\n".join(lines)

SYSTEM_PROMPT = (
    "You are a professional smart-contract security auditor for bug bounties (Immunefi). "
    "Given a Solidity contract and a function, find the vulnerability, prove it with a "
    "concrete exploit PoC, and give the exact fix. Output ONLY strict JSON:\n"
    '{"vulnerable": true/false, "vuln_type": "...", "severity": "critical|high|medium|low|gas", '
    '"poc": "<step-by-step exploit sequence>", "fix": "<what the fix is and why>", '
    '"patched_function": "<the FULL replacement function including its signature, '
    'or null if not vulnerable>"}'
)

SEVERITY_MAP = {"LOW": "low", "MEDIUM": "medium", "HIGH": "high", "GAS": "gas"}

def normalize_severity(s):
    return SEVERITY_MAP.get(s, (s or "low").lower())

def build_context(r, max_chars=MAX_CONTEXT_CHARS):
    '''Contract context: full source when available, else the vulnerable-function
    excerpt, else the audit narrative — truncated to fit the context window.'''
    if r.get("source"):
        ctx = r["source"]
    elif r.get("function"):
        ctx = r["function"]
    else:
        ctx = r.get("audit_text", "")
    return ctx[:max_chars] + ("..." if len(ctx) > max_chars else "")

def build_messages(r):
    user = (f"Contract `{r.get('contract_name') or 'unknown'}`:\n\n"
            f"{build_context(r)}\n\n"
            f"Audit this contract. Finding id: {r['id']}.")
    if CHECKLIST_CONTEXT:
        cc = checklist_context(r.get("vuln_type", ""), r.get("audit_text", ""))
        if cc:
            user += "\n\n" + cc
    answer = {
        "vulnerable": bool(r.get("is_real", True)),
        "vuln_type": r.get("vuln_type", ""),
        "severity": normalize_severity(r.get("severity", "")),
        "poc": r.get("poc", ""),
        "fix": r.get("fix", ""),
        "patched_function": None,  # corpus fixes are prose, not full replacement functions
    }
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": json.dumps(answer, ensure_ascii=False)},
    ]

def extract_json(text):
    '''Same parser as scripts/eval_harness.py: first {...} block, json.loads.'''
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("no JSON object in model output")
    return json.loads(text[start:end + 1])

def score_answer(gold, pred):
    '''gold/pred: strict-JSON dicts. Returns per-key correctness.'''
    gv, pv = (gold.get("vuln_type") or "").strip().lower(), (pred.get("vuln_type") or "").strip().lower()
    return {
        "json_ok": True,
        "vulnerable": gold.get("vulnerable") is pred.get("vulnerable"),
        "severity": gold.get("severity") == pred.get("severity"),
        "vuln_type": bool(gv) and (gv == pv or gv in pv or pv in gv),
    }

# ---- build training/eval record lists (drop the 56 empty-vuln_type records) ----
train_records = [r for r in train_rows if r["vuln_type"].strip()]
eval_records  = [r for r in eval_rows if r["vuln_type"].strip()]
print(f"train records after drop-empty-vuln_type: {len(train_records)} (dropped {len(train_rows) - len(train_records)})")
print(f"eval  records after drop-empty-vuln_type: {len(eval_records)} (dropped {len(eval_rows) - len(eval_records)})")

# sanity: every answer is strict JSON and round-trips through the harness parser
for r in train_records[:3] + [train_records[len(train_records) // 2]]:
    msgs = build_messages(r)
    gold = json.loads(msgs[-1]["content"])
    assert extract_json(msgs[-1]["content"]) == gold, "answer does not round-trip through extract_json"
    print("example answer:", json.dumps(gold, ensure_ascii=False)[:200])


In [ ]:
# 6) Tokenizer + datasets — chat-template the records; SFTTrainer tokenizes lazily.

from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_for_sft(record):
    return tokenizer.apply_chat_template(build_messages(record), tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_list(train_records)
eval_ds  = Dataset.from_list(eval_records[:EVAL_MAX_ROWS])
print(f"train_ds = {len(train_ds)} rows | eval_ds = {len(eval_ds)} rows")

sample = format_for_sft(train_records[0])
print("formatted example chars:", len(sample))
print(sample[:700].replace("\n", "\n"))


In [ ]:
# 7) QLoRA setup — 4-bit NF4 base + LoRA adapters (all linear layers).

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "CUDA GPU required (Kaggle: Runtime -> Change runtime type -> GPU T4 x2)"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print("compute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",  # flash_attention_2 needs sm_80+; T4 is sm_75
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


In [ ]:
# 8) Train — QLoRA SFT with the corpus split as validation. (Kaggle T4: ~1-2 h.)

from trl import SFTTrainer

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
)

try:  # modern trl: SFTConfig carries max_seq_length
    from trl import SFTConfig
    args = SFTConfig(**sft_kwargs, max_seq_length=MAX_SEQ_LEN, packing=False)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        formatting_func=format_for_sft,
    )
except ImportError:  # older trl: TrainingArguments + SFTTrainer(max_seq_length=...)
    from transformers import TrainingArguments
    args = TrainingArguments(**sft_kwargs)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        formatting_func=format_for_sft,
        max_seq_length=MAX_SEQ_LEN, packing=False,
    )

trainer.train()
TRAINED = True


In [ ]:
# 9) Post-training eval — strict-JSON accuracy on held-out eval_source.jsonl.
#    Scoring reuses the harness contract: extract_json() then per-key match.
#    (The final gate is scripts/eval_harness.py + Echidna on the VPS.)

def run_eval(model, tokenizer, records, n=EVAL_N_ROWS, max_new_tokens=MAX_NEW_TOKENS):
    sample = random.Random(SEED).sample(records, min(n, len(records)))
    results = []
    for r in sample:
        msgs = build_messages(r)
        gold = json.loads(msgs[-1]["content"])
        prompt = tokenizer.apply_chat_template(
            msgs[:-1], tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(
            prompt, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)
        try:
            row = score_answer(gold, extract_json(text))
        except Exception as e:
            row = {"json_ok": False, "vulnerable": False, "severity": False, "vuln_type": False,
                   "error": f"{type(e).__name__}: {str(e)[:120]}"}
        row["gold_severity"] = gold["severity"]
        if row["json_ok"]:
            row["pred_severity"] = extract_json(text).get("severity")
        results.append(row)
    return results

if "TRAINED" in globals() and TRAINED:
    results = run_eval(trainer.model, tokenizer, eval_records)
    n = len(results)
    print(f"evaluated {n} held-out records")
    for k in ("json_ok", "vulnerable", "severity", "vuln_type"):
        print(f"{k:12s}: {sum(r[k] for r in results)}/{n} = {100 * sum(r[k] for r in results) / n:.1f}%")
    bad = [r for r in results if not r["json_ok"]]
    print("non-JSON outputs:", len(bad), bad[0].get("error", "") if bad else "")
else:
    print("skip eval — model not trained in this session")


In [ ]:
# 10) Save the adapter (weights + tokenizer + config) — then merge/export on the VPS.

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("adapter saved to", OUTPUT_DIR)

# Optional GGUF export for Ollama/LM Studio (docs pipeline; needs Unsloth + more RAM):
#   !pip install -q unsloth
#   merged = model.merge_and_unload(); merged.save_pretrained(OUTPUT_DIR + "-merged")
#   # convert to GGUF, then: ollama create hunter -f <(vps_setup.sh Modelfile)


## After training

1. Download `/kaggle/working/hunter-qlora` (adapter + tokenizer).
2. On the VPS: merge the adapter into the base model, export GGUF (Unsloth), import into Ollama
   (`scripts/vps_setup.sh` scaffolds the whole stack: Ollama + Foundry + Echidna + Slither + Modelfile).
3. Gate candidates: `python3 scripts/eval_harness.py --model hunter --contract x.sol --function f`
   — `GATE=PASS` only when the patched contract compiles **and** Echidna observes a behavioral divergence.
4. Manual PoC review → Immunefi submission.

**Checklist wiring:** the user prompt includes the Findings Checklist items matched to the record's
`vuln_type`/`audit_text` (see `docs/findings_checklist.json`, `scripts/checklist_map.py`,
`docs/checklist_vuln_type_map.json`) when `CHECKLIST_CONTEXT = True`. Train and eval prompts are built
by the same `build_messages()` — no train/eval prompt skew.

**Known corpus gaps (documented in ANALYSIS_REPORT.md):** no negative examples (`is_real` is always true),
`poc`/`fix` empty on ~80% of rows, `patched_function` is not trainable from the corpus (fixes are prose).
The eval cell therefore scores `vulnerable`/`severity`/`vuln_type`; the Echidna gate handles patch validation separately.
